## 1. Detect zipped or automatically unzipped input


In [ ]:
from pathlib import Path
import base64, json, os, subprocess, sys, time, zipfile
INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
RUN_ROOT = WORK_ROOT / "jcam_thm09_fixed_regularity"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_OK = True
STAGES = []
def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)
mode = "embedded_exact_sources"
if INPUT_ROOT.exists():
    if any(INPUT_ROOT.rglob("check_fixed_regularity_information.py")):
        mode = "kaggle_auto_unzipped_detected; embedded exact sources selected"
    elif any(INPUT_ROOT.rglob("*.zip")):
        mode = "kaggle_zip_detected; embedded exact sources selected"
record("bootstrap", "passed", mode)

## 2. Install exact source package from notebook cell


In [ ]:
SOURCE_ARCHIVE = base64.b64decode("".join([
    'UEsDBBQAAAAIANozQ12DwANG5AQAAC8MAAA2AAAAQ29kZS9leHBlcmltZW50cy9jaGVja19maXhlZF9yZWd1bGFyaXR5X2lu',
    'Zm9ybWF0aW9uLnB5pVZtb9s2EP7OX3HQPkxKbc32EKzN6gGDO7TFNnRw061AEAiMdLbZSqJAUrHbof99d6TkWLbXdpgBSyT1',
    '3NtzdySjKHqGDk2lamWdyqFuKzQqlyUUSq5rzYsWVtrA9Yvfx5MnENfagYTGaL1K0iiKhFgZXUGWrVrXGswyUFWjDYFqgkqn',
    'dG2F6NZye98P31ldB9FGuk2p7nq5P2gqhFi+enUNcz+LSbkqSXWSGrS6vMc4SRtpsHb2ZnYrXr1hpBf4DqKFLjDiAWHb0lk/',
    'XqkdFpnBdVtKo9yHTNUUVOXdi1hBWr0vlIk7rfNr0+IIcEesZPq9nyZCLMjMJL0US/+ezi4FG57OHovX2Z8///bml9c0jSfp',
    'ZHI5IsBk6p+zMPFPv0KCiXibPV++fEZ41zYlxoq8nE4mE8+1AlWDkfUaY1qbJol485/QQhS4glzfo5FrzKysSKiId1ewKrV0',
    'I3AbdLKbJTD+KYyuBNCvxBUhjFpvHNtjJIxh2QnBI1h6mEFKdw2V3MV+zr/BpF+g0EcQ70jHXkECFxcwo5W4PV1ORl9UMT6v',
    'YnxGRbIfMVUtUxW43K+rlQ+ZpKc4ns7g6Zxg9AgMPAqrIugSwuitJVpubgXrs6yvz32gz7uSBRLDuKdyQSbsiF6PwHoo7mTu',
    'sgZNZnRbFwSZwQUs6W8Zw2+P26rCbfhrerlf+wZe1gU2SI/alR9AN05V6iOyTXBbPd5qUxbQpR64wYlJ0HfvMHfqHkHX1KGw',
    'G6+NKlKvkkdZD5yzyDCz8ZK8knf2IRE+zCRwTgk6/e5D7wAJ54fjG2Rkxwy+fchI0kW32GD+3gejSYmqaUeybZO17Fa2gwod',
    'bVO0SRkKh8JnVb/OpyAdFWZDnU+sSB9noxU1dIiwYyNDY7Txadz7wn6fNkyXQnb9NLo+rmE8fZ9QWHG8GB0SRRQtBkXBK5P0',
    '+2PQJP1hCAsmbjtqXm8RGyh0JclGCA+cBofWecKwvsdSN/it5bSpqq1CBQUO2Mmc2EHjXVyE3l74/rvah9Iz1ZfeSWfHPRVB',
    'V8IFu+9JRh9/97VPjRxAA13n6mDYu9Ja5COFdA49G4fQEnh60KcH6EFNj4877qwYOz8slCGMd4BUNtx6D5T8PQgostEVhTtc',
    '844+2CaEXzlCBQ87jzPc5WjtQOgohCPxI0Eu6MDUqSQdIeccOGTsnP3D78fGSaJnbt9LijcoPnE9l6TgDL9HesJhXarG5hvl',
    'PmYtkc2SMzoGziGtM7pek0kq/B2ZOgvd0D7ykQ77K7h++PKp39e3ioop5ovEFy4LKV1hoiSl9qrjaBuNoMZtqWqcRzTGOteF',
    'qtfzqHWr8eMooZoCvrqExtqSMmq7OV+D0mcqd3/5hZgRI8JhWdSyQjsv6doRc53dTG675g+yqX9tUBYkdvqBRbwcRWTbqpLm',
    'A1kLtRnl0iLXZUmee0ygIZJlmeW821KeuQU4yf76I/qK6svA6VBLmWEqOJXKq7r5TNnd+u4mkL+kHJg9gJ/qpRL5P3o/X3xB',
    '81fU6r/aoIstkrLoOfUCBO74RnyHocdAlmu8MxLuWkf7NPCVmT7TYdTFQyZ+BIsYrtGZ21STJ2lVpNFIfBJfU4VZl92Ub9FU',
    'jj77mcOdC1sSL6dFWzW0XwbkiGLg6OZ0OTqpU5GIxtA5En9WLhH/AFBLAwQUAAAACADIM0NdX9NjlnQHAABIDwAAGgAAAE1h',
    'dGgvcHJvb2ZzL3Byb29mX3RobTA5Lm1kfVfdjts2Fr73UxDoReyx5cl4O4tuWl2kbYoGzWKD7XRbYJBINEWPOZVElaTGduO5',
    'b19g74u2LxBggVznAeYh8iT7HZL6mbTolWWKOjw/3/edww/YxZf/TB7+4xGTey4c26i9LBIjr9qSG+UOTNVOmhteJqreaFNx',
    'p3TNGqOEZPjPdC2Zbl3TOlZIoSzeTiYnJ1877lr7iD3/97/+8+Rzv9NtJbNYlgXb8EqVB8brYjCva6xspCzWXHy/PDlhz6Rj',
    '+Xfp5cPF2Yt8wXKRnp2u6MHg4SM80Of5w0/su59/o4Ule8y2qigk3OOGVxKGWX735t1PP12KxC7E3L7ImbIhRO8SL0uWX+TM',
    '6LYu7JJdwEW306yEl7Vjen0thVM30jJu5GSSb7Ozl3dvpvtZOt0nd2/mZvbSe7TNVuP1xK8vJ5PHjskbaQ7hAJ+BUnJTwzEr',
    'YdWnBJ4yoaumhMN9OtimrQWl2npHgxUEkz+CG8+ya3/WcZ+I7PqY2Fk2f/uaHPny/pu5ndH6ZLLbSiORwuwsFYnJQ+pEtkrF',
    '3OQ+bispRu9Ro+HFTmGlUrWq2spvr/jeP+sNxXsd4s2ZhmfhK2kq5ai6Q/a7aPwJB3+A0PWN3C+YNgVcKkIVr5Bjhn2qoExU',
    'ANiQh4iIBZZE2RaqvvJlk3Xh/WRXhhcK1bKntl33f0ItO5+FxnGqRlXZThVuSyjIv02n56ermc0/hv8tFr7JLuLSBXLyqca+',
    'PwUCWyXWGV1fAbAhHB8Ed6zS1uHtM9VYsVXuR7ADEM7J90I2cBnG8BGl0BI+PMTv/pchl6/2t8jmOlTAU/Kc7bSxLnHcXGHf',
    'c5zsdGLxKCjp/Apg0taSNc4s8lL2TPSZiQSB+bOXJ9O7N7MUDM5e7cGH725Hh/pU1f7cIAG+6nyPfw2iRkFPTuReSGtBSmWB',
    'P7LzmNm2yV7dp9cte4IXd79mjl12J2TudpYMTrxgKbuYroyd27evZ/gzPTs9ezhf2dPzGUMFKC1PPYOdEqRBSJ6zzJDuMKeZ',
    'aCusUi0GjISalgrlWUu3k4gmJ6sR5387XeFxt1WljCJExUtC6UjiaNdQs7UXA2ZkxVUdxCKiKeaFl1ca0ritYroBHUSZinxE',
    '1UB4MlwDwJbVukcy+6HFjmCy1DvQZB02l/hHAFsrZzjZwOe6Uj/K91Wyd8D6E3BiPShIAwwmnRT3NEk6PendINRLBQ2MMhPj',
    'HSGVkEX5iqJK6rMkbX9utN6QQndy2dMjpAgem0CUnLQwVoHkEkywOiBts1GCmMoI0KCjhx5iyy/pmwXtfgFofkEqHdHtiE2A',
    'LKz4MrYNfCZFIgkaDAa+2PuiRVsGFbBeZlonl+wJF9tRANw5ZMH2yodQNDGbMKh39RAgEgUv8GY4uCvSgoFPgvrFwLIUnrzq',
    'msbb14uuT7x9fZtOzfxIf48k1gR/nNXWCjAZH0coS1GCBaklqeCY2Ya+9PX4kwbC1wgf2CR8sfWhM+qVtMtR+OvT+kOL0hWs',
    'UGjWNXV5oyucnRP73PiE/BI9hpifXVNrHSNaglqlbiS5/QeDsMM74eyN8dgf26Z573s3EnJCJADXiz9MFyQNwnpkrUmzhy6z',
    'ZN+CJj5XhzS2xAAe4D6IhoSFEqn90B4CffPDu59/txGz00PooP2b3yDb7Gvloxi5Bc5ZtYa85LBCKA1Diy8ahpa5Sc+HiaWb',
    'Vxb3TAx96UOLJvTRLD23GHcw1JQILD9OQ8+dPUDXPcLCanxQtNzveZD68eOLOHWNVca3B6PwpuI2uOctSc+DDfE0SzAk2J62',
    '2RxTAsX9lERBCRTrD90Zxr4Hq+KYAjXM3xcSyC8O6AuH1lYWbMfBxK22sl50KlYXilAL2zr0pF7sHgQGNhw9tpPGMI2So3pt',
    '4dJIKm0Qj+BSGPly3/ti4p1R3PdMVXsQUCOgOYQa3NTzMUuOgZjZ/Dg7XTEAg1l8HTpJPp2GTQDIPDzO8YiNNAQKWkWHIzAF',
    'Vl/QCAYcxRmLSpBUktc0QSehXfR9HTCAsmBURKg5uD2P/ZLA167RHYTr6BrnJY01OLUpNSHV+GnQtlVF27z4YcwNKOH+jH59',
    'aGYhpzX6PLNb6kRe3no/aNAPaXyviFtuh1Fh1J7DzDAKZGj7iOMzYqixsjyEcgTTcX7p+imZzo/4Q+ro2TdqscPAGfyIChKT',
    'PJzuh7K8y2AAus83Kqk6CfBjgAut6V54fpJEPRqjb+LEfn9KUp4V7uBLY/HUBsUJt5RAIposz+0FUc1DLEZMKPeDzZK9++8v',
    '/tokIHqhtSpqyQ1XaKV+HPw78M/R43FAIRMLGtIxAuWE316fvkrP8kdRSDC2YuqA0voGNp59/AVsdL8D5UiX0XPwc+jmnae4',
    'zmlJcws6HF6sS2W3pBZf5aw7m8fb4tga/CDwROyE204Y2TuYqSqmMvK7H234jpQ7WBSSVj/+y3sR+onAABziCh/gTpN738Mw',
    'Y0OfwbzXenAS0wpVydoGhQnXUN85arhUxlFwbTQvElGSPvIGGRd8rTxCDLCi4KSVdMeBW/KGyi/kcvJ/UEsDBBQAAAAIAOQN',
    'Q13uk1mjdwkAAEYTAAAaAAAATWF0aC9wcm9vZnMvcHJvb2ZfdGhtMDYubWR1WN2O28YVvtdTDOCLrmRRKwlu2jhVACe1kcW6',
    'iNHdJgYMWxyRQ2nWJIeeIVeSvQaKoiiSXjZA4dsi6QsUKODrPsA+xD5Jv3NmSGrjGEhgaTk8c36+7zvn6I44/+oP0fST+0KW',
    'Qpe1spcyj3SZGVvIWptSVFYnSuC7yHSpayWeSKtqEzlVC9PUVVMPBqPRWS3rxt0XT/749TcPf8/H640SDn9WqVhps7pQSa0v',
    'lXjVyNTCdCKSXDqHe9P+YlPme5Epla5k8nIyGonzjXbCKtfktcAnucKVQooRbh8JkwlZi8K4WsSnsZAJOexEpaywpoFdsi3F',
    'WpWNLhUsJ6bMco1j5VrUWxP1XvmgOkcm4qQWqVFOlKYmR3UBQ8hIsqF34b9PicxzJMisclU44RCVy/Z04MFvbv78w4PZdDIY',
    'PEae4qeLZ9Px7Hk8FnGxmNM/p+Pzm+//PaOP8vMp/iFfY7uYHc+n8QRxK9GUL0uzLcXW2DwVlbSyUHCP8hBfv7/57rtndjyL',
    '7HMcf1ALdansPsSdKtRKcQVyuFqiUm2kbjCIN8vZi+v3R7vhQh7touv3d+3wBTkl8GR+60nETxAF+ZMraUvKrUoUWRJcLboE',
    'uURuiyqHf30x8X+D4vdJzpoylCh+tLygW5AV+TwWR1yoci9UmVYGBo4L7T+ItZWpRgRu+Bld5ZRApYRO8SedyBy3cRE4R47L',
    '5kEJrAiCaHy2rG/+/penMaElkTbVpcx1vb+FnJVCMVUPPMFFe7i8fr94RkkYU47gJ7nZwAW1A9YQO3Pn179ybZZxYZQYFEKu',
    'kS7jKNnX/4WZo7PhwjXV8k2DspHdt3A8W77Z4evZWwBrt3xzsZiN52+fbZYXvgRR+NQMny/vxi2SYO70xejo+v1wwRauzq5u',
    'vv/p9K1o76FouCickshVKtEZ2GYyQB+gQLyRT62sAN2dDjxX1ho7EY+A6tGIwQT6WURsCv1a/ZyjLRZwkaV8aMcZz/QORz1g',
    'gdFYuCbZ4JCESCCfMl+L6x+Xb+rF7O2Lc/Es+Lysh1Ef13MBYgh5fnz02+npUCzEn5bhM0FUHM2Gg8GWr62ZJJq0SmyVXm/q',
    'Qz+3Oq03JDdACupNrIGlhTwHX05K4hMQ1OTSjok7gFFG5HLNqrYySATse9GLWF9u5SvLjcGrvyxdQheVcYpSQkmXFpIaHDqQ',
    'zw4qlQIo6/3Ey10mCw1bG0nqI2oFUxZIT63O6on4wsDIB6xmVsRzGUeutqZce7G7VDuvK/Tgsa4c5Kt+Dd5CkWLmk7aiQIRU',
    'YuuNpCilBk/5RX9CZQCQDgBnGYZjZaogrIqlsJdN0omWxAd8J8PwB+2g9Ia9cwD1nTviiTUm8xJDQtLrBXsmfWhEQ0/AmMgY',
    'e6TGu9/xgzG4fMklM+1Rq9ImQWJWlK8+UZ8Jh2hRdaoWW/g82PtW1xB3ER8ynuqbWCUd2Y53cfsVdj04rDuoAruXqsMTTiHS',
    'dCK+UmXiARvq3+cUL2hHiArJbcWF9Yd4T4GSOMakFk/hE+RNxOmCBNprP9ntDdbSrkkqmgWej1FQCDb6w66SJd+z1wpaCUKS',
    '7FyIDxWHSLi8C+bJo/TF/O7cXqVXQ98FGic6RQNN59IyYPDKGNIThPP6HYtQS01cSQ62IBHxyaLtXCTKuSrXKFH8eDGL5lSI',
    'L1QiWWW5o/WUat+nLHcYa/sDZe4jXWHMTSDmE0AtyJYyeTL2NLAfiuLdbJWtoRJ6/StVmFMqiXL3XaItjFc9321isQWObnUX',
    'dhRekHVTqijVheJKwFHSBuXqqCT5WsGe16vXXmMgHfjO7WodejdA8S6k/B1Szlr5+Pjo3ikX6AShqTSokqOy06XfGEiC0SJR',
    'ee447tbN+ITEMAs895UIWnLz1x+Ws/FkMhnTp1c0qLyiThNzIH1J16RNsEk+SQa5EUpC9ek2tMZElWFoQU6QXEoKWdT/+8/x',
    'PYjQlxItYo9x6SzZbKV97SNFaX5cahEOIsjHOH4ac4PPTJ6brevbyhukAq3F98B33E98D7HIzByZYaYfDEgT6iPzYUstAX1N',
    'csUGBc1cPGAR0XKZqIMaUggUdzud+GTiMZeJekcLUv/EM1g85JvjtvfHLO3S22RVgdkufR1yLMdLHipNbbbXmbEXNX6TBLGJ',
    'uWTbIGAgEd2u4V6n7WPhDHL1kekC2fpJ3JP2ahc1V+0Xn/DB4EvoNuJDGUmU+j6NY90pcRbaJtKDSnfM93ltKhrJVzyaZtYU',
    'lHs41OAAMiPi8zD+EunbEXhMgvDp8Wwa34ffbSWjcCNr06fH96bTaIYz06H/UxgTwrwqQ3f1l5AoISuQQSQRdFW7hGjd9eBD',
    'gHIQsoDDNSUO/xUqsJynGlnXUqOtafTjRzRQ5vtxwFenUlRj3/FjGeJD58GIvAuziuMMgC3QhDGhnntMN6Lc/PNf3B1P+l0s',
    'KkyqvP65xFRoqRuVvIQqRL7vUuNoR4PbvXc0wh5D8xyclDnC8MOObOnwqIFaEvLxjDKW9XG0SusOBoJuKMeCp/GmN8vLEu9H',
    'fk2LGfxQl+AS7wMgxuH7W5rjMBlXuSbd4sRDpHkGWTfabeRK+3HdrpuCrGxNg+EykzqfhLi70a9DOw97DXWp7XBBnfxou5xH',
    '2+WMRkjU05fK86/jqeBNs4Ick2KlbbgYziu2Zw9HKGSNQMUiGffkDa0uFNqPQzz32Iar4edlXjb8tqjbOUESwiDTddT54HsA',
    'V5tnusgPTdwNFBBZIwZQBut1u4mwPIJrheZy7Xkh8fsQ+uHaWChE8fEGRbvuReNowYYLGTylmNZWp5Tqh9A49ggTuB8CRyPo',
    'cMds2iE8pz5YgzwFKc4Vmt3PN5N+I6GtfuT9oMgFjV25iiDD+3ZDX8tqLDZ7BHlpciBiTHiSpDi4wgGvfmjc8Y4+6Te0pLGW',
    '4HOoRd2Cz4s9+xfPIGeiHRISRYF/S+2u44PntNtYXb6k2oFBa+L4zd/+MfW9MVi/lCXwq5xHgcNYwtTIgb9kI8t1qHwgRw+t',
    'Xyx4OzxjftUHv0xoZ+j94NR2o6G2L5Wq/PaCDaR/0esXxkXa1AuIif/5xb9Jy7u0sFZ6dHits2pN+9Gt+/HHgrUP+lNSdk48',
    '5Fl8Ih9M1M1sl3hbkohC49C/HKWQMWQyP0CT7hxsMX6CY1b4Diyp1VM0H8VMgDf3lX7iDj+EmBIjRRr2Kkh/zq770aX/iQu8',
    'WOvOp35hQ2NqIe35/pHfqQb/B1BLAwQUAAAACACbJkNdPsnILI4EAABHCAAAGgAAAE1hdGgvcHJvb2ZzL3Byb29mX3RobTA3',
    'Lm1kZVXLbhs3FN3rKy6QRSRFI8sG2gQutHDRBAmUoEatdmMkIjXDkejMDKckx5YdByi6SrotUHRbtP2EAv4KfYS/pOeSM1Ka',
    'AoZMcMj7OPecwwc0f/4qmTw+plL6dK2rFRmbKUu5seTXKn7+ktY6y1RFp9IqbyiXpS6ue73h8MxL37hjOv3u2x+efhNuqY1M',
    'fXEdbseDJKuMdOWVvZRFYirs1NZ4k5qCTN6mGA+H9FJ5Eq8W72aj+XtBSxVilLrSpdwgbq1SrzIaqk2qnBtS2pRNIb2+VOSU',
    'T1JzqaxcKSqMc6Ner4s01VW+OCHX1It327v7Dx/O7egwsa/f01Nsb/9c+PPtP4vtXf9s4QcJlrM3w/72bvBaIMjVWllF4kSQ',
    'ldVKOeIkvM5MqW/U533JYmWs9uvS0RX+kfRUGoeuZoJM4+vGU21wxVHNYUxTZSMSdnp4cDQRowAU9+yvDJnlBfpFcy5s77or',
    'lbc6JUyCpEP6Dj+ar1WFrr9fzA/6TyazAd1//ItaDMKav8yQJVTG56ZyLsa93nzduP9ADcCMBSG0qTApnaqWDtrRcGiV4wJ4',
    'EpEVyZWxRdbOGlNco6xAgpZKYvt7P6QeCC4SQfBXGU+SVqpCUwWdPL7/6deTw8mehU3NAC0jQBWyS4z1qtuKLeR6gyKsWoEF',
    'AP0aGFXOS0ZXMv4Zzlzii3Jo8sEDOrXG5Gg3UnoVoOW2Y2BdqR8bWXCcQGPAff3poHejHRGA3mGiGC1dNmVkxp4AyPmsPRGb',
    '2ScYUeMiuTMF9jDszut0RLlS2VKmbxNdZapW+KlAIJ0FztDK6ozE2fQd9fUjMGZwMKNj0tPJaDwej2bJIUE1YFxbOgM1pme7',
    'XkQD7j8F0e9/+fl8MjoEv8mZEuze4MOZIIeJuxxwkbjdJM0tOHN40D8Kc/vaAEyUn1kcSj8lp1gvLt5s70RgpDiSInmpa4cp',
    '+hsCf0RMdcztahspBjyto0Izdt0BrgV5+zHa4GH/ZnA7PZK3N0m6uOBSEHlMzxl5kBwsXVycx7P9zSBpV83g9eJR4Dpucgth',
    'LcF6prl8y8z638gEWmeFwSCs2u82gq61KjJ02NrDAMFCLDrTTACx94r7j3+zfqMzMb93wucLpGS67gZy1pRl8NmQZS5aIgoJ',
    'hUw7iTaVZgnCURghoNsqJ1fS6WWh9jxjK+2I3On3c76N6f63P4IEXrBboU0fxN3rJewa7fl9yOhULrgWdIyT0YMq0JOlS3Bd',
    'ZqZZOva+eAA86+gLoqggO3qBKDvnQv37voKOIStws0LxplmtQw/RTNghmLWkWQIapMzGbbE5KjI2eTKhlaz5IBDaCX8UjCX4',
    'TjBEvBHWcqk1a58yo6L1tGHjKxXNqgOvi8UNmdpjr8Dolo2LAoxlIO1nDyYU7do3D8H8WvoYG2lseJ/4qavxMoXxqTzXqebC',
    'vLQr5SNnoBeVZLpUlQOkyNs9Lnw3ul2mZcmeAWgrMAyhOw/NrQkABLi5rqopl6wzntxOr7QX594tOxdSn7gVdK6+im/LFw8j',
    'FTpLAvUhFKkx9MIYp8a9fwFQSwMEFAAAAAgAby1DXazGUIrhAwAA6AYAABoAAABNYXRoL3Byb29mcy9wcm9vZl90aG0wOC5t',
    'ZGVUS47bRhDd6xQFZBFJFmWPgXxgY1aJgxijOINICAYwbLFFFskek91Md3NG8mgWzsrJAbIPkivkGs4d5iR51aTmgwACxP5V',
    'vXrvVX1Gq+9/SJ58/YwK5nyjsndJ4Zip4eB0lhgOdKocB5t4fGb2gp0qeTSaTpdBhc4/o9Offvz5xbfz6ZQWuJGepbRhMtZw',
    '04YdKZPjVdOqLMziwndtaz0Tq6yiWgU2gezmnLOgL5jSah3OU9Ke0kWaLHTrs0qH92SNRJ5RZ3RhXVPvSGMnzM7TeZ92efP7',
    'r0idWRMUjlSgxnrsn6TUWm2Cj8krhRyxCG1KcirXHTL9+2G8fH42OQa09VV38/Hj2bU2xfpqi8/l9X6/Tbr9Hom+sw5RdsRF',
    'oTMtwMEJ3r9Yh7dVzD+jUHHP6BdymBwIG/gkr4L2hWY/GqWf/lpfheOj67cr+vTPOoyXE7r57W9a0ICHVul8NPrGGs+/dMhW',
    '72akC4G7PhkDbsS4X+7x6OR6eDMg2DBqfygomEgiE+SDA+3lDmz4W0WhGJWOceAQQYHcxZAHKGiFmF41TBAPFza2A5mqbWtU',
    'QsESbzP294LJPylxgeFSRWUvravzxLecabB3D05RW+t6ctOz49dPZkdvhioanfdXSqeRLoiyvq//+Ojx+OnJJH0uPoh8f/m5',
    'p0I1GtZIF8dPVYwBH5VI7u9k+epBCalaPT4RlqfTU2dtIS4WINGcqVjhvrZZZcW5qncUpdEfqXgt1z4ok/Gd7w560HhwvxGC',
    '2PjOAU5fSwNRZ4R0nUSFad1GQxsnNfhG1TX5GvpNenIYnO7uWmVG6WtplvF2ksT/bvJm/QheWAyGlc8Dil5BGEY3XdOrkwry',
    'vh8d3+7eWTsoV4q7cerZQcDoK2FtTsvDdSdrj+jqnTSU1NC2zm51AyOhCtsGfNZ3LSdDRLLWHIKsNTLEIulCGe2re3IdfJ7C',
    'sIgmZM3plQUDggemQmOgIUyQVX3o/J5TLC54q8Mu0gvDW+S63QTRTA5NpR2jmps//hQDvJRIqDbE0OKElfgHP0Ub5eHZku3Q',
    'xlhzrQ2iKEAdOkb8fsE14me10s2cXhqq7WWSa4DyiAmUuW3ExMCNYYo3GJVNB69FuQ+B+vkBP2edc6JFq5wc10nNyhlcKzvs',
    'AC8jCRiUfqacW4YYMiqFvlvKTdds8MQWcbvoYCvYEXAGMDO6rDQgDHAgX6XhQ+lZ9Faly4puK5gfZlujdhG7BZNwL6LKaFL/',
    'qxd8Fowa0BpSRKTvfST4vlNkOD+cqsQAneecCwYVUTyIO1Tg0VkcDSVGgsDShvLV1+Ir5UAKRB79B1BLAwQUAAAACADsM0Nd',
    'VEDZ8L0FAAAsDAAAFQAAAE1hdGgvdGhlb3JlbV9pbmRleC5tZIVWS3LbRhDd6xRTlY2kEPrZjp1iecFITMwS9Sl9HO+EIdAE',
    'Jx7MIDMDSkppkZwhm5wjR8gBkjvkJHk9AAHKkp2FSgAx05/Xr1/3V+JqQdZRKZTJ6W5j40FMjsSDuCBf64CHyyBD7fEw8r4u',
    'q6Cs4bcjqggXTKaIX0/tknS4F6qsrAvSZCQeNh6SJHn2D06u3p0ke/u4Ob0WygvYqqwyQVTO5nUWhHU5OXw+vzh7P+aA3kut',
    'cgQZyC2lbkKYK6NWAU3t7VDMbW1yyT9JLTo3B3z43shSZaJUraOzH47EjI+LWxUWItzaRNss3sUveVj4de+j/X9//X30in9y',
    '9ifK+NhAZNYs6U74elY4mSsyIf4WpDIlXnD6xCINGUhIL6SYSU9aGRoKrT6SvhcVuVC7WeNVuqJurq0CfwELkzbjxBpcmBPl',
    'M5l9RKYuA/Lp9c3V7kEq6K5CUJQLR4WjAJzgjnO6tU4zbL4tylpKLwcdnI+NDwROs7G1axOTKx+UKWrlF3KmtEK1Yc0GRKyk',
    'xvUMT07qAZILnHTtSCzJqblqYTWwD6tdei9h4MwAmwgnvGW1Q0oxodoopFgmlaM5OeIg2tRiuegOlxjruVYZhyXsLJZlSY/q',
    'dvkkkZEuaOYkU7zWOGzrIKQR4K/zwVlYyrRUZR8lF/17JholszovEMG5RBw28cQR4J4sqKXS3NlSVBIoaNI91zRJZ2D/MaMG',
    'YvRisye+dGFrEKsyes1k298THX8lbK+ii5z7AvozTqjr1WRBcnkfs5mO25YbR/D2j8TPtcxRKLRFk1LHh+jwllSxCEkpKzFV',
    'lc8WKvzC7hnK8AzIvbU5Wk1zgIe2rDTF+nh8djQQpTU2WKOyJoXY1rjuhb83YUF8vYW9LxqCH52fN8FfqhLCJA1ZqBKiyVXb',
    '7pWtat0QbYbwPbeIGH84TwCktgWzN8OD9zE51CzxGYRPFMhvLZkfQS9lEvRkthATaOG6D9iupA/DptzIGZAbz0TF85wi5/0Q',
    'oXjFVBSFs3XFmjZrWqYRy3eW5vMcoAxEVDA+p/IBc56Dj0xCxlWlV50zw/cCsiHXNBgwtu0F3zhzfTgR2QIcscj2vufvN7AV',
    'ZWLzzd7xVtr3vDIceONA21vIbUti4La9TXfQF7+9/Szbo7ysd99Mdf33HA8adAdoYbBYRaFCl3lAZawoGWo2UhCaBChHwXyi',
    'fLXW/ZjopSo9TgXHhj6u6uAZ0juYX6gc00k08td1cpdASRzH8y2E+jllXYKGfKxfPPE4zg7a16zwq+jTv//YZJgZ4xJlLeWd',
    'aEDs3TZTjQEG1Vfl+V+UmPWyjDOkuTJEvSnJFaaFb4iZ2xJDZyjSA5kKdAGPkb5pV5RqHUb+rxS2VykmYexmyG2B+MCxFpG1',
    'iJI22iiSw65kXQ1j7SyQ6lF6wwLa1iuZO1qhnxiUDaVLPhHRdPrPbzfHmx+2rtI1UFae+qwKsmxopTOsaemHlCcwYFYGgdwq',
    'Tys00mma9Hf7afGEQV9iy3fS43/rGE/9QM8wQ1CE6FZgjjn22dopoHyQTKIek287GU6P3+6nDWkTTDhImGNl/IRB6dXmgfNf',
    '+7/+3Hq7ub+7v/f1gd99tQW+pc8TJwqT/rwux8Wlk3vCq0bVnm2fJ2vCZzoRnGwxFM2nzwE5fLSZPLtXXFBpeZSvGIsxokKd',
    's2ZjYPAYlEvIbWyliGBPRS4nM9y0K2lHxsnp+2aETGLyEn1ji5qahQLla9YQOdMEELAMJE23WiczzXp0Oh4fXYrR0dHkanJ2',
    'OpqK0eXl9ck5v3RGeVhmTMag2E40XVc84Nbp2k51zAU7x1wnHkCXxING8veNjVMbVzBMuAWWH16QZZdP7LwdMVpfs2SNEcXH',
    'gD8PTZYpLJnYZ7UwWCoBOp6WzayNu5YHWTJOkEsg0p2d3UOsqrtX78ZnF+OTm9Prk/HF5HA0vRldI9+dMk93Nv4DUEsBAhQA',
    'FAAAAAgA2jNDXYPAA0bkBAAALwwAADYAAAAAAAAAAAAAALaBAAAAAENvZGUvZXhwZXJpbWVudHMvY2hlY2tfZml4ZWRfcmVn',
    'dWxhcml0eV9pbmZvcm1hdGlvbi5weVBLAQIUABQAAAAIAMgzQ11f02OWdAcAAEgPAAAaAAAAAAAAAAAAAAC2gTgFAABNYXRo',
    'L3Byb29mcy9wcm9vZl90aG0wOS5tZFBLAQIUABQAAAAIAOQNQ13uk1mjdwkAAEYTAAAaAAAAAAAAAAAAAAC2geQMAABNYXRo',
    'L3Byb29mcy9wcm9vZl90aG0wNi5tZFBLAQIUABQAAAAIAJsmQ10+ycgsjgQAAEcIAAAaAAAAAAAAAAAAAAC2gZMWAABNYXRo',
    'L3Byb29mcy9wcm9vZl90aG0wNy5tZFBLAQIUABQAAAAIAG8tQ12sxlCK4QMAAOgGAAAaAAAAAAAAAAAAAAC2gVkbAABNYXRo',
    'L3Byb29mcy9wcm9vZl90aG0wOC5tZFBLAQIUABQAAAAIAOwzQ11UQNnwvQUAACwMAAAVAAAAAAAAAAAAAAC2gXIfAABNYXRo',
    'L3RoZW9yZW1faW5kZXgubWRQSwUGAAAAAAYABgDHAQAAYiUAAAAA'
]))
(RUN_ROOT / "source_package.zip").write_bytes(SOURCE_ARCHIVE)
with zipfile.ZipFile(RUN_ROOT / "source_package.zip") as archive:
    for member in archive.infolist():
        target = (RUN_ROOT / member.filename).resolve()
        if not target.is_relative_to(RUN_ROOT.resolve()):
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(RUN_ROOT)
record("sources", "passed", "experiment, proofs, manuscript source and PDF")

## 3. Run six fixed-regularity comparisons


In [ ]:
if RUN_OK:
    started = time.perf_counter()
    command = [sys.executable, str(RUN_ROOT / "Code" / "experiments" / "check_fixed_regularity_information.py")]
    result = subprocess.run(command, cwd=RUN_ROOT, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-5000:])
    if result.returncode:
        RUN_OK = False
        record("experiment", "failed", f"exit={result.returncode}")
    else:
        summary_path = RUN_ROOT / "Code" / "results" / "fixed_regularity_information" / "fixed_regularity_information_summary.json"
        summary = json.loads(summary_path.read_text())
        RUN_OK = bool(summary["cases"] == 6 and summary["all_checks_passed"]
                      and summary["max_coverage_identity_error"] < 1e-10)
        record("experiment", "passed" if RUN_OK else "failed",
               f"six cases; seconds={time.perf_counter()-started:.2f}")

## 4. Zip outputs and sources


In [ ]:
(RUN_ROOT / "notebook_stages.json").write_text(
    json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
FINAL_ZIP = WORK_ROOT / "result_jcam_thm09_fixed_regularity.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file() and path.name != "source_package.zip":
            archive.write(path, arcname=(Path("thm09_fixed_regularity") / path.relative_to(RUN_ROOT)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)